# AI Agent Memory: What to Store and What to Throw Away

Everyone races to make agents remember **more**. The agent that wins keeps the right things and throws the rest away. Store everything and memory gets expensive, slow, and dirty ([Demo 05](../05-memory-hygiene-demo/) shows it is also dangerous); store nothing and you are back to the amnesiac agent of Demo 01.

The capability that decides this is **selection** (memory extraction): what to keep, in which memory *type*, and what to throw away. This notebook builds it three ways and measures them against the same planted conversation.

The first two mechanisms run on Strands' [native memory framework](https://strandsagents.com/docs/user-guide/concepts/memory/overview/): no hand-rolled memory tools, no memory logic in the chat agent's system prompt. You attach a [`MemoryManager`](https://strandsagents.com/docs/api/python/strands.memory.memory_manager/) with `Agent(memory_manager=...)`; it registers a `search_memory` tool, runs a [`ModelExtractor`](https://strandsagents.com/docs/api/python/strands.memory.extraction.model_extractor/) off the turn to decide what to keep, and injects recalled memory into the model. You own two things: the extractor's **selection prompt** and the **store**.

| Mechanism | What it is | Who owns the policy | Where it stores |
|-----------|------------|---------------------|-----------------|
| **A: native, 1 store** | `MemoryManager` + one `MemoryStore` | you (one prompt) | one vector partition |
| **B: native, 4 typed stores** | `MemoryManager` + four `MemoryStore`s | you (one prompt per type) | one partition per type |
| **C: Amazon Bedrock AgentCore Memory** | fully managed by AWS | AWS (managed, or override) | AgentCore's managed store |

**A vs B is granularity**, not backend: A is the simplest native setup (one store, one prompt); B reproduces AgentCore's per-type partitioning with four typed stores and four specialized prompts, so the built-in criteria become text you own. **The vector backend is a separate lever**: both A and B run on Amazon S3 Vectors *or* Amazon DynamoDB Vector Search via `VECTOR_BACKEND` (the two backends from [Demo 02](../02-vector-memory-demo/)). C is the fully managed counterpart: send raw turns, AWS extracts.

The ground truth is planted: **5 items to keep** (2 facts, 2 preferences, 1 episode) and **3 decoys to throw away** (small talk, a passing opinion, ephemeral weather). Scoring is deterministic (no LLM judge) on **selection recall**: did the keepers get stored (kept / 5)? Storing everything would ace recall while hoarding the junk, so the decoys confirm a mechanism is selecting, not hoarding.


## Install dependencies

In [1]:
%pip install -q -r requirements.txt


[notice] A new release of pip is available: 26.0.1 -> 26.2
[notice] To update, run: /opt/homebrew/opt/python@3.10/bin/python3.10 -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


## Step 0: Credentials, permissions, and your own resource names

This demo **creates AWS resources in your account** (it doesn't just read them), so you need AWS credentials with the right permissions, and you must pick names you own.

**Required, AWS credentials** (`aws configure`, or `AWS_PROFILE` in `.env`) with these permissions:
- **Amazon Bedrock** (always): `bedrock:InvokeModel` on `amazon.titan-embed-text-v2:0`, the embedding model the store uses for semantic search. This is required no matter which chat model you pick.
- **Amazon S3 Vectors** (`VECTOR_BACKEND=s3`): `s3vectors:CreateVectorBucket`, `CreateIndex`, `PutVectors`, `QueryVectors`, `ListVectors`, `DeleteVectors`.
- **Amazon DynamoDB** (`VECTOR_BACKEND=dynamodb`): `dynamodb:CreateTable`, `DescribeTable`, `UpdateTable`, `PutItem`, `SearchVectors`, `Scan`, `DeleteItem`.
- **Amazon Bedrock AgentCore** (mechanism C): `bedrock-agentcore-control:CreateMemory`, `GetMemory`, `ListMemories` and `bedrock-agentcore:CreateEvent`, `RetrieveMemoryRecords`.

**Optional, the chat model.** The conversational agent and the `ModelExtractor` (mechanisms A & B) run on OpenAI `gpt-4o-mini` by default, which needs `OPENAI_API_KEY` in `.env`. You can skip OpenAI entirely and use **Amazon Bedrock** for the chat model too, uncomment the `BedrockModel` line in the Mechanism A cell. Either way, Bedrock is still required for embeddings.

**What gets created, where, and with what name:**

| Resource | Created in | By (code) | Name |
|----------|-----------|-----------|------|
| S3 Vectors bucket + index | the step that uses it (A / B) | `memory_stores.py` → `S3VectorStore._ensure()` | bucket `VECTOR_BUCKET` + index per partition |
| DynamoDB table + vector index | the step that uses it (A / B) | `memory_stores.py` → `DynamoDBVectorStore._ensure()` | `<DYNAMODB_TABLE_PREFIX>-<partition>` |
| AgentCore memory (4 strategies) | Step 5 (mechanism C) | `agentcore_memory.py` → `ensure_memory()` | `AGENTCORE_MEMORY_NAME` |

Each store self-provisions the first time a mechanism instantiates it, so there's no separate setup step: you'll see a "created / ready" line inside each mechanism.

> ⚠️ **S3 bucket names are globally unique across ALL of AWS** (not per account). You cannot reuse the example name; the next cell derives a unique one from your account id (override it with your own in `.env`).


In [2]:
import os

# Bearer-token env vars would override the AWS profile: drop them before boto3 loads.
os.environ.pop('AWS_BEARER_TOKEN', None)
os.environ.pop('AWS_BEARER_TOKEN_BEDROCK', None)

from dotenv import load_dotenv
load_dotenv()   # reads .env (OPENAI_API_KEY, optional AWS_PROFILE, VECTOR_BUCKET, ...)

assert os.getenv('OPENAI_API_KEY'), 'Set OPENAI_API_KEY in .env (see Step 0 above).'

import boto3
os.environ['AWS_REGION'] = os.getenv('AWS_REGION', 'us-east-1')
_profile = os.getenv('AWS_PROFILE')
_session = boto3.Session(profile_name=_profile) if _profile else boto3.Session()
_ident = _session.client('sts', region_name=os.environ['AWS_REGION']).get_caller_identity()
_account = _ident['Account']

os.environ['VECTOR_BACKEND'] = os.getenv('VECTOR_BACKEND', 's3')      # 's3' or 'dynamodb'

# S3 bucket names must be globally unique. If you didn't set VECTOR_BUCKET in .env,
# derive one from your account id so it doesn't collide with anyone else's.
os.environ['VECTOR_BUCKET'] = os.getenv('VECTOR_BUCKET', f'agent-memory-demo-{_account}')
os.environ['DYNAMODB_TABLE_PREFIX'] = os.getenv('DYNAMODB_TABLE_PREFIX', 'selective-memory')
os.environ['AGENTCORE_MEMORY_NAME'] = os.getenv('AGENTCORE_MEMORY_NAME', 'SelectiveMemoryDemo')

print('These resources will be CREATED (or reused) in:')
print(f'  AWS account : {_account}')
print(f'  AWS region  : {os.environ["AWS_REGION"]}')
print(f'  backend     : {os.environ["VECTOR_BACKEND"]}')
if os.environ['VECTOR_BACKEND'] == 's3':
    print(f'  S3 bucket   : {os.environ["VECTOR_BUCKET"]}   (globally-unique name)')
else:
    print(f'  DynamoDB    : {os.environ["DYNAMODB_TABLE_PREFIX"]}-*')
print(f'  AgentCore   : {os.environ["AGENTCORE_MEMORY_NAME"]} (mechanism C)')
print('\nOverride any of these by setting them in .env before running.')

## Step 1: Preflight (resources are created where they're used)

There is no bulk 'create everything' step. Each `VectorMemoryStore` self-provisions its index (S3 Vectors) or table (DynamoDB) the moment a mechanism instantiates it, and AgentCore's memory is created in its own step (Mechanism C). So you'll see a `Provisioning...` line inside each mechanism, creating only what that mechanism needs, in the account and backend shown in Step 0.

The cell below is just a preflight: it confirms your backend choice and that the vector service is reachable, without creating anything.


In [3]:
import memory_stores as ms

# Preflight only: confirm the backend and that the vector service answers.
# Resources themselves are created lazily inside each mechanism (see the
# 'Provisioning...' line in Steps 3, 5, and 6), so nothing is created here.
print(f'Vector backend: {ms.VECTOR_BACKEND}')
if ms.VECTOR_BACKEND == 's3':
    print(f'Target S3 Vectors bucket: {ms.VECTOR_BUCKET} (created on first use by Mechanism A/B)')
else:
    print(f'Target DynamoDB tables: {ms.DYNAMODB_TABLE_PREFIX}-* (created on first use by Mechanism A/B)')
print('AgentCore memory (Mechanism C) is created in its own step.')
print('\nPreflight OK. Each mechanism will provision only what it needs, when it runs.')


## Step 2: The planted conversation and its ground truth

Every mechanism is graded against the **same** six turns from a brand-new user. Three turns carry information worth **keeping**; three are **decoys** a good selector must ignore. Because we know the right answer in advance, scoring is deterministic: no LLM judge.

In [ ]:
import json, time

# The 6 turns. Three carry information worth KEEPING; three are DECOYS to ignore.
CONVERSATION = [
    "Hi! I'm Sam. I'm vegetarian with a severe shellfish allergy.",              # keep: vegetarian + shellfish allergy
    "Gorgeous weather out here today, hope your day is going great!",            # decoy: small talk
    "For flights: I refuse overnight layovers, and I keep fares under $1,500.",  # keep: no layovers + $1,500 budget
    "I watched a documentary about airplanes last night, it was okay I guess.",  # decoy: passing opinion
    "I just booked the Iberia flight JFK to Madrid for October 10th!",           # keep: Madrid booking
    "Apparently it might drizzle here later this afternoon.",                    # decoy: ephemeral weather
]

# Ground truth: a good memory contains every KEEPER word and none of the DECOY words.
KEEPERS = ['vegetarian', 'shellfish', 'layover', '1,500', 'madrid']
DECOYS = ['gorgeous', 'documentary', 'drizzle']

def score(text):
    """The score is selection recall: how many keepers got stored. We also track
    which decoys leaked in, as a check that a mechanism is selecting rather than
    hoarding (a store that keeps everything would ace recall and still be useless).
    Raw stored count is not a quality metric: more is not better."""
    text = text.lower()
    kept = [w for w in KEEPERS if w in text]
    leaked = [w for w in DECOYS if w in text]
    return (f'selection recall: {len(kept)}/{len(KEEPERS)} kept {kept}\n'
            f'  decoys kept out: {len(DECOYS) - len(leaked)}/{len(DECOYS)}'
            f' (leaked: {leaked or "none"})')

# Show the planted conversation so it's clear what each mechanism is graded against.
print('The planted conversation (6 turns):\n')
for turn in CONVERSATION:
    print(f'  {turn}')
print(f'\nMust KEEP these words appear in memory: {KEEPERS}')
print(f'Must NOT appear (decoys):              {DECOYS}')


### The score: selection recall

The conversation mixes 5 keepers with 3 decoys to throw away. The score below is **selection recall**: how many of the 5 keepers a mechanism stored. The decoys are there so a mechanism cannot win by hoarding, keeping everything would ace recall and still be useless. The raw stored count is shown for transparency, not as a score.


## Step 3: Mechanism A: native `MemoryManager`, one store

This is the simplest way to give an agent long-term memory with Strands: attach **one** [`MemoryManager`](https://strandsagents.com/docs/user-guide/concepts/memory/overview/?trk=87c4c426-cddf-4799-a299-273337552ad8&sc_channel=el) and let it do the work. Before the code, here's what each piece is and why it exists.

### The mental model

A chat model, by itself, forgets everything when the turn ends. To remember across turns and sessions, three separate jobs have to happen, and the `MemoryManager` orchestrates all three so you don't wire them by hand:

1. **Extraction**: *deciding what to keep.* After the conversation, something reads the turn and pulls out the durable bits ("vegetarian", "no overnight layovers") while throwing away the chit-chat.
2. **Storage**: *writing it somewhere that survives.* The kept facts are saved to a store outside the agent's short-term memory.
3. **Recall + injection**: *bringing it back when relevant.* On later turns, related memories are searched and fed back to the model.

### The pieces you assemble

| Piece | What it is | Who owns it |
|-------|-----------|-------------|
| [`MemoryStore`](https://strandsagents.com/docs/api/python/strands.memory.types/?trk=87c4c426-cddf-4799-a299-273337552ad8&sc_channel=el) | **Where memories live** and how they're searched. Here it's our `VectorMemoryStore`, backed by Amazon S3 Vectors, so recall is *semantic* (by meaning, not keywords). | you (or a vended store) |
| [`ModelExtractor`](https://strandsagents.com/docs/api/python/strands.memory.extraction.model_extractor/?trk=87c4c426-cddf-4799-a299-273337552ad8&sc_channel=el) | **How selection happens**: a *language-model* call whose system prompt IS your keep/discard policy. (Note: this is a reasoning model like `gpt-4o-mini`, *not* the embedding model, since embeddings happen inside the store.) | you write the prompt |
| `IntervalTrigger` | **When extraction runs**: e.g. every turn. It fires the extractor *off* the conversation path, in the background. | you pick the cadence |
| `MemoryManager` | **The orchestrator**: ties the above together, auto-registers a `search_memory` tool, and injects recalled memory into the prompt. | the framework |

### Two models, two different jobs (this trips people up)

- The **language model** in the `ModelExtractor` *reads and decides* what to remember → outputs text like `[{"content": "vegetarian"}]`.
- The **embedding model** (Amazon Titan Text Embeddings V2, defined in `memory_stores.py` and called *inside* the store) turns that text into a vector for semantic search. You won't see it in the cell below: it's encapsulated in `VectorMemoryStore`.

### What you DON'T do

The chat agent gets **no memory instructions**: its system prompt is just its persona ("You are a flight assistant"). You don't hand-write memory tools, and you don't burn tokens telling the chat model to remember. You own exactly two things: the **extractor's prompt** and the **store**.

> **On timing:** the synchronous `agent("...")` call flushes extraction before it returns, so the extractor's model call is part of the turn's latency and the memory is queryable the instant the turn ends. (With the async APIs, extraction stays in the background and you flush at shutdown, see the `flush` note later.)

In [ ]:
# OTEL_SDK_DISABLED silences OpenTelemetry tracing noise in notebook output.
os.environ['OTEL_SDK_DISABLED'] = 'true'

from strands import Agent
from strands.models.openai import OpenAIModel
# The native Strands memory framework: this is all of it:
from strands.memory import MemoryManager, ModelExtractor, ExtractionConfig, IntervalTrigger
from vector_memory_store import VectorMemoryStore
import memory_stores   # holds the embedding model id

# TWO models are involved, doing DIFFERENT jobs:
#  1. A language model (below) powers the ModelExtractor: it reads each turn and
#     DECIDES what to keep. This is reasoning/selection: NOT embeddings.
#  2. An embedding model (Amazon Titan Text Embeddings V2) runs INSIDE the store
#     (vector_memory_store.py -> memory_stores.embed): it turns each kept memory
#     and each query into a vector for semantic search.
MODEL = OpenAIModel(model_id='gpt-4o-mini')          # language model: chat + extraction
print('language model (chat + ModelExtractor):', 'gpt-4o-mini')
print('embedding model (inside VectorMemoryStore):', memory_stores.EMBED_MODEL_ID)
# Amazon Bedrock for the language model instead: comment above, uncomment below.
# from strands.models import BedrockModel
# MODEL = BedrockModel(model_id='openai.gpt-oss-120b-1:0', region_name='us-west-2')

# The selection policy: the ONLY thing you write. Returning [] is 'discard'.
SELECTION_PROMPT = (
    'You extract durable memories worth keeping about a traveler, from a transcript.\n'
    'KEEP: durable facts (name, allergies), stated preferences (cabin, layovers, budget), '
    'and notable events (a booking, a cancellation).\n'
    'DISCARD: small talk, weather, passing opinions, questions.\n'
    'Return ONLY a JSON array of {"content": string}, or [] if nothing is worth keeping.'
)

# One store whose recall is SEMANTIC (it embeds with Titan V2 internally). It is told
# HOW to select (ModelExtractor, a language model) and WHEN (IntervalTrigger, off the turn).
store_a = VectorMemoryStore(
    name='traveler_memory',
    partition='selective-single',
    extraction=ExtractionConfig(
        trigger=[IntervalTrigger(turns=1)],
        extractor=ModelExtractor(model=MODEL, system_prompt=SELECTION_PROMPT),
    ),
)
store_a.clear()   # rerun-safe

# Attach the MemoryManager to the agent. The chat prompt has NO memory logic.
agent_a = Agent(
    model=MODEL,
    system_prompt='You are a helpful flight assistant. Be concise: 2-3 sentences max.',
    memory_manager=MemoryManager(stores=[store_a]),
    callback_handler=None,
)

turn_ms = []
for turn in CONVERSATION:
    t0 = time.perf_counter()
    agent_a(turn)   # the MemoryManager extracts + stores automatically, off the turn
    turn_ms.append((time.perf_counter() - t0) * 1000)

# Read memory back through the manager's native semantic search.
# NOTE: in a notebook use `await` (a loop is already running); a .py script uses asyncio.run().
entries = await agent_a.memory_manager.search('dietary restrictions preferences bookings')
print('\nstored memories (count is transparency, not a quality score):')
for e in entries:
    print(f'  - {e.content}')
print()
print(score(' '.join(e.content for e in entries)))
print(f'turn latency: {sum(turn_ms)/len(turn_ms):.0f} ms/turn (incl. extraction) | available immediately')


## Step 4: Where does this memory live, and how do I get it back?

Unlike the old `agent.state` (a dict in RAM that dies with the process), a `MemoryStore` persists **outside** the agent. Here `VectorMemoryStore` writes to **Amazon S3 Vectors** (or DynamoDB): a bucket/index in your AWS account, not this notebook's memory. So it **survives restarts**: close this notebook, reopen it tomorrow, and the memories are still there.

There's no `agent.state` to print, but the store exposes its own inspection: `store.count()` (how many), `await manager.search(...)` (semantic recall), and the raw index behind it. The cell below shows all three, then proves recovery by opening a **fresh** store (as if from a new session) and searching it.

In [ ]:
# 1) HOW MANY memories are stored, and WHERE they live.
print(f'store name : {store_a.name}')
print(f'lives in   : Amazon S3 Vectors: bucket {memory_stores.VECTOR_BUCKET!r}, index "selective-single"')
print(f'memories   : {store_a.count()}')

# 2) The RAW stored memories (text + cosine score), straight from the vector index.
import memory_stores as ms
print('\nraw contents of the vector index:')
for text, sim in store_a._backend.query(ms.embed('traveler profile and trip'), top_k=10):
    print(f'  ({sim:.3f})  {text}')

# 3) RECOVERY from a fresh session: a brand-new store object pointed at the SAME
#    partition sees everything the previous run wrote: because it's in S3, not RAM.
print('\n--- simulating a new session (fresh store, no conversation replayed) ---')
reopened = VectorMemoryStore(name='traveler_memory', partition='selective-single')
print(f'reopened store sees {reopened.count()} memories')
recovered = await reopened.search('what should I avoid eating on this trip?')
for e in recovered:
    print(f'  recalled: {e.content}  (score {e.metadata["score"]:.3f})')

## Step 5: Mechanism B: four typed stores (one memory *type* each)

Mechanism A (Step 3) used **one** store with **one** general prompt: everything worth keeping lands in the same place. Mechanism B splits memory into the **four types** this series uses: `facts`, `preferences`, `trip_summary`, `episodes`: giving each its **own store**, its **own vector partition** (a separate S3 Vectors index), and its **own selection prompt**. This is how the managed service (AgentCore, Step 6) partitions memory internally; here you build the same shape yourself and *own the prompts*.

**How each store is built (3 ingredients):**

1. **A partition**: the S3 Vectors index this type's memories live in (e.g. `selective-facts`). Separate indexes mean a search for a preference never wades through episodes.
2. **A selection prompt**: tells the `ModelExtractor` what to KEEP for *this type only*. The `facts` prompt keeps allergies but rejects preferences; the `preferences` prompt does the opposite.
3. **The shared output contract**: every extractor must return the same shape. Rather than repeat it in all four prompts, we define it once as `JSON_CONTRACT` and **append** it (`prompt + JSON_CONTRACT`). Each extractor's final prompt = *its own rules* **+** *the shared format*.

The cell below reads the `TYPED` table and builds one `VectorMemoryStore` per row.

In [ ]:
# Part 1: Define the four memory types. Each row is: type -> (vector partition, selection rules).
# The 'selection rules' are the per-type keep/discard policy: the part a managed
# service hides from you. Here they are plain text you can read and tune.
TYPED = {
    'facts':        ('selective-facts',   'Extract durable FACTS (identity, home airport, allergies). '
                                          'NOT preferences, NOT opinions, NOT small talk, NOT weather.'),
    'preferences':  ('selective-prefs',   'Extract stated PREFERENCES (cabin, seats, layovers, budget). '
                                          'NOT facts like allergies, NOT events, NOT small talk.'),
    'trip_summary': ('selective-summary', 'If the turn advances the CURRENT trip, return one summary sentence. '
                                          'Otherwise return [].'),
    'episodes':     ('selective-episodes','If the turn is a notable EVENT (a booking, a cancellation), '
                                          'describe it in one sentence. Otherwise return [].'),
}

# Part 2: The shared OUTPUT CONTRACT. Every ModelExtractor must return the same JSON
# shape, so we write that rule ONCE here and append it to each type's rules below
# (prompt + JSON_CONTRACT) instead of repeating it four times.
JSON_CONTRACT = ' Return ONLY a JSON array of {"content": string}, or [] if none.'

# Part 3: Build ONE store per type. Each store gets its own partition and its own
# extractor whose system prompt = (this type's selection rules) + (the shared contract).
stores_b = {}
for mem_type, (partition, selection_rules) in TYPED.items():
    full_prompt = selection_rules + JSON_CONTRACT          # per-type rules + shared output format
    stores_b[mem_type] = VectorMemoryStore(
        name=mem_type,
        partition=partition,                               # its own S3 Vectors index
        extraction=ExtractionConfig(
            trigger=[IntervalTrigger(turns=1)],            # extract every turn, off the turn
            extractor=ModelExtractor(model=MODEL, system_prompt=full_prompt),
        ),
    )

# Show one fully-assembled prompt so the 'prompt + JSON_CONTRACT' is concrete:
print('Example: the facts extractor system prompt:')
print('  ' + stores_b['facts'].extraction['extractor']._system_prompt)

for s in stores_b.values():
    s.clear()   # rerun-safe

# Part 4: Attach ALL FOUR stores to one MemoryManager. The manager routes each
# extractor's output to its own store; the chat prompt still has no memory logic.
agent_b = Agent(
    model=MODEL,
    system_prompt='You are a helpful flight assistant. Be concise: 2-3 sentences max.',
    memory_manager=MemoryManager(stores=list(stores_b.values())),
    callback_handler=None,
)

# Part 5: Run the same conversation and see how memories split across the four stores.
turn_ms_b = []
for turn in CONVERSATION:
    t0 = time.perf_counter()
    agent_b(turn)
    turn_ms_b.append((time.perf_counter() - t0) * 1000)

print('\nstored per type (count is transparency, not a quality score):', {name: s.count() for name, s in stores_b.items()})
entries = await agent_b.memory_manager.search('dietary restrictions preferences bookings')
print(score(' '.join(e.content for e in entries)))
print(f'turn latency: {sum(turn_ms_b)/len(turn_ms_b):.0f} ms/turn (incl. extraction) | available immediately')


## Step 5b: Same mechanism, DynamoDB backend

Mechanisms A and B ran on Amazon S3 Vectors (the default). The backend is a lever, not part of the mechanism, so here we run the **same typed-store setup on Amazon DynamoDB Vector Search** instead, with one line: flip `VECTOR_BACKEND` to `dynamodb`. The vectors now live inside DynamoDB tables (`selective-memory-*`) rather than an S3 Vectors bucket. Same Titan embeddings, same selection prompts, same scores; only *where* the vectors live changes.

Requires `boto3>=1.43.72` (the `SearchVectors` API). The tables are created on first use and removed by the Step 7 cleanup cell.


In [ ]:
# Flip the backend to DynamoDB for this cell. make_store() reads the module-level
# ms.VECTOR_BACKEND, so we set BOTH the env var and the module attribute.
import importlib
os.environ['VECTOR_BACKEND'] = 'dynamodb'
ms.VECTOR_BACKEND = 'dynamodb'   # take effect without re-importing
print('backend for this cell:', ms.VECTOR_BACKEND)

# Rebuild the four typed stores on DynamoDB (same TYPED prompts as Step 5).
stores_ddb = {
    mem_type: VectorMemoryStore(
        name=mem_type,
        partition=partition,
        extraction=ExtractionConfig(
            trigger=[IntervalTrigger(turns=1)],
            extractor=ModelExtractor(model=MODEL, system_prompt=prompt + JSON_CONTRACT),
        ),
    )
    for mem_type, (partition, prompt) in TYPED.items()
}
for s in stores_ddb.values():
    s.clear()

agent_ddb = Agent(
    model=MODEL,
    system_prompt='You are a helpful flight assistant. Be concise, 2-3 sentences max.',
    memory_manager=MemoryManager(stores=list(stores_ddb.values())),
    callback_handler=None,
)

turn_ms_ddb = []
for turn in CONVERSATION:
    t0 = time.perf_counter()
    agent_ddb(turn)
    turn_ms_ddb.append((time.perf_counter() - t0) * 1000)

print('stored per type on DynamoDB (count is transparency, not a quality score):',
      {name: s.count() for name, s in stores_ddb.items()})
entries = await agent_ddb.memory_manager.search('dietary restrictions preferences bookings')
print(score(' '.join(e.content for e in entries)))
print(f'turn latency: {sum(turn_ms_ddb)/len(turn_ms_ddb):.0f} ms/turn (incl. extraction)')

# Restore the default backend so later cells and reruns behave as before.
os.environ['VECTOR_BACKEND'] = 's3'
ms.VECTOR_BACKEND = 's3'


## Step 6: Mechanism C: AgentCore Memory (fully managed)

Send the **raw turns** (`create_event`): no selection on your side at all. The four built-in strategies (semantic / userPreference / summary / episodic) extract, embed, and index asynchronously inside AWS. The memory (with all 4 strategies) is created by `ensure_memory()` if it doesn't exist.

Two API facts learned by running this (not in the docs): the episodic strategy **requires** `reflectionConfiguration.namespaces`: a bare `{'name': ...}` fails validation, and a memory in `CREATING` status can't be deleted; wait for `ACTIVE`.

Extraction is async, so we poll and **measure the lag**: a number AWS doesn't publish.

In [6]:
# agentcore_memory wraps the two AgentCore clients + self-provisioning.
import agentcore_memory as acm

info = acm.ensure_memory()      # creates the memory with 4 strategies if missing
memory_id, sids = info['memory_id'], info['strategy_ids']
actor = f'sam-{int(time.time())}'   # fresh actor -> clean namespaces per run
session = 'selective-run'

turn_ms_c = [acm.send_turn(memory_id, actor, session, 'USER', t) for t in CONVERSATION]
print(f'6 raw turns sent | turn latency: {sum(turn_ms_c)/len(turn_ms_c):.0f} ms/turn (create_event only)')

lag = acm.wait_for_extraction(memory_id, sids['facts'], actor,
                              'dietary restrictions travel preferences', timeout_s=420)
print(f'time until queryable (measured, not published by AWS): {lag:.0f}s')

stored_c, per_strategy = '', {}
for name, sid in sids.items():
    scoped = name in ('tripSummary', 'episodes')
    records = acm.retrieve(memory_id, sid, actor, 'traveler profile and trip',
                           session_id=session if scoped else None, top_k=10)
    per_strategy[name] = len(records)
    stored_c += ' '.join(records) + ' '

print('records per strategy (count is transparency, not a quality score):', per_strategy)
print(score(stored_c))

6 raw turns sent | create_event avg: 347 ms


extraction lag (measured): 85s


records per strategy: {'episodes': 0, 'facts': 1, 'preferences': 2, 'tripSummary': 1}
kept 5/5 ['E1', 'F1', 'F2', 'P1', 'P2'] | decoys leaked: ['D2', 'D3']


## Step 7: Cleanup (optional)

Delete the resources this notebook created: the S3 Vectors indexes/bucket (or DynamoDB tables) and the AgentCore Memory. Self-provisioning re-creates them next run, so this is optional.

In [ ]:
import threading
# Full teardown: deletes EVERYTHING this demo can create, on BOTH backends,
# regardless of which VECTOR_BACKEND you ran. Safe to run repeatedly (idempotent).
import boto3, os, time
from dotenv import load_dotenv
load_dotenv()

AWS_REGION = os.getenv('AWS_REGION', 'us-east-1')
session = boto3.Session(profile_name=os.getenv('AWS_PROFILE')) if os.getenv('AWS_PROFILE') else boto3.Session()

PARTITIONS = ['selective-single', 'selective-facts', 'selective-prefs',
              'selective-summary', 'selective-episodes']

# ── Amazon S3 Vectors: empty each index, delete it, then delete the bucket ───
bucket = os.getenv('VECTOR_BUCKET', f"agent-memory-demo-{session.client('sts').get_caller_identity()['Account']}")
s3v = session.client('s3vectors', region_name=AWS_REGION)
try:
    existing = {i['indexName'] for i in s3v.list_indexes(vectorBucketName=bucket).get('indexes', [])}
    for idx_name in existing:
        keys = [v['key'] for v in s3v.list_vectors(vectorBucketName=bucket, indexName=idx_name).get('vectors', [])]
        if keys:
            s3v.delete_vectors(vectorBucketName=bucket, indexName=idx_name, keys=keys)   # empty first
        s3v.delete_index(vectorBucketName=bucket, indexName=idx_name)
        print(f"  deleted S3 Vectors index '{idx_name}' (and its vectors)")
    s3v.delete_vector_bucket(vectorBucketName=bucket)   # delete the bucket itself
    print(f"  deleted S3 Vectors bucket '{bucket}'")
except s3v.exceptions.NotFoundException:
    print(f"  S3 Vectors bucket '{bucket}' not found: nothing to delete")

# ── Amazon DynamoDB: delete each table (delete_table removes all its items) ──
prefix = os.getenv('DYNAMODB_TABLE_PREFIX', 'selective-memory')
ddb = session.client('dynamodb', region_name=AWS_REGION)
for p in PARTITIONS:
    table = f'{prefix}-{p}'
    try:
        ddb.delete_table(TableName=table)   # removes the table AND all its items/vectors
        print(f"  deleting DynamoDB table '{table}' (and all its contents)")
    except ddb.exceptions.ResourceNotFoundException:
        pass

# ── Amazon Bedrock AgentCore: delete the memory (waits out CREATING) ─────────
memory_name = os.getenv('AGENTCORE_MEMORY_NAME', 'SelectiveMemoryDemo')
ctrl = session.client('bedrock-agentcore-control', region_name=AWS_REGION)
memory_id = next((m['id'] for m in ctrl.list_memories(maxResults=50).get('memories', [])
                  if m['id'].startswith(memory_name + '-')), None)
if memory_id is None:
    print(f"  AgentCore Memory '{memory_name}' not found: nothing to delete")
else:
    for _ in range(30):   # can't delete while CREATING
        if ctrl.get_memory(memoryId=memory_id)['memory']['status'] != 'CREATING':
            break
        print('  AgentCore memory still CREATING: waiting...'); threading.Event().wait(10)
    ctrl.delete_memory(memoryId=memory_id)
    print(f"  deleted AgentCore Memory '{memory_id}'")

print('\nTeardown complete: every resource this demo can create has been removed.')
